# DIM_CARATER_ATENDIMENTO

Caráter da internação (campo `CAR_INT` do SIH), com a descrição oficial (`CARATER.CNV`).
`TIPO_CARATER` agrupa os códigos: `01` = **ELETIVO**; `02` (urgência) e `03`–`06`
(acidentes, lesões e envenenamentos) = **URGENCIA**; `00` = **NAO INFORMADO**.

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_CARATER_ATENDIMENTO d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_CARATER = s.SK)
WHEN NOT MATCHED THEN INSERT (SK_CARATER, COD_CARATER, NOME_CARATER, TIPO_CARATER)
VALUES (-1, '-1', 'NAO INFORMADO', 'NAO INFORMADO')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_CARATER_ATENDIMENTO d
USING (
    SELECT c.CODIGO AS COD_CARATER,
           NVL(t.DESCRICAO, 'NAO CADASTRADO - ' || c.CODIGO) AS NOME_CARATER,
           CASE WHEN c.CODIGO = '01' THEN 'ELETIVO'
                WHEN c.CODIGO IN ('02', '03', '04', '05', '06') THEN 'URGENCIA'
                ELSE 'NAO INFORMADO' END AS TIPO_CARATER
    FROM (SELECT CODIGO FROM SILVER_DOMINIO WHERE TABELA = 'CARATER'
          UNION SELECT COD_CARATER FROM SILVER_SIH WHERE COD_CARATER IS NOT NULL) c
    LEFT JOIN SILVER_DOMINIO t ON t.TABELA = 'CARATER' AND t.CODIGO = c.CODIGO
) s
ON (d.COD_CARATER = s.COD_CARATER)
WHEN MATCHED THEN UPDATE SET d.NOME_CARATER = s.NOME_CARATER, d.TIPO_CARATER = s.TIPO_CARATER
  WHERE DECODE(d.NOME_CARATER, s.NOME_CARATER, 0, 1) + DECODE(d.TIPO_CARATER, s.TIPO_CARATER, 0, 1) > 0
WHEN NOT MATCHED THEN INSERT (SK_CARATER, COD_CARATER, NOME_CARATER, TIPO_CARATER)
VALUES (SEQ_DIM_CARATER.NEXTVAL, s.COD_CARATER, s.NOME_CARATER, s.TIPO_CARATER)
"""

In [ ]:
mesclar("DIM_CARATER_ATENDIMENTO", SQL, origem="(SELECT CODIGO FROM SILVER_DOMINIO WHERE TABELA = 'CARATER')")

In [ ]:
consultar("SELECT * FROM DIM_CARATER_ATENDIMENTO ORDER BY 1 FETCH FIRST 10 ROWS ONLY")